# 08 - 手写 generate (KV Cache + PD 两阶段)

> 配套笔记：[01-推理基础.md](https://github.com/Zoey-Cheng/MLSys-Learning-Notes/blob/main/notes/04_推理优化/01_推理基础.md)
> 基础：[01-mini-llama.ipynb](https://github.com/Zoey-Cheng/MLSys-Learning-Notes/blob/main/code/01-mini-llama.ipynb) ｜ [Colab](https://drive.google.com/file/d/1_R9oORTHsXZTkbW9OEUmRz8azSxLGtmW/view?usp=sharing)

**核心展示** 对应到 doc §3.2 + §5 + §6 的三块手写 code：

- §1 `MHA_KV` ← doc §3.2（MHA with KV cache）
- §3 `sample` ← doc §5（采样 pipeline）
- §4 `generate` ← doc §6（PD 两阶段 + batch EOS）

**支持运行的辅助** 部分（让上面三块能装起来跑通 / 验证 / 套真实模型）：

| ipynb 节 | 角色 | doc 对应 |
| --- | --- | --- |
| §1 `MHA_KV` | 🌟 核心 | §3.2 |
| §2 装配 `DecoderBlock` + `MiniLlama_KV` | 辅助：让 §1 能跑通 | 01-Transformer篇 |
| §3 `sample` | 🌟 核心 | §5 |
| §4 `generate` | 🌟 核心 | §6 |
| §5 Smoke test（4 组） | 辅助：用 toy model 验证 doc 论述 | §1.1 / §3.1 / §1.2 / §6 |
| §6 真实 Llama 系小模型（HF TinyLlama） | 辅助：把 §4 接口套真实模型 | bonus，doc 不展开 |

## 0. 环境

In [1]:
!nvidia-smi

Mon Jun 22 06:43:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install x-transformers rotary-embedding-torch -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.6/119.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.7 MB/s eta 0:00:00


In [3]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from x_transformers import RMSNorm
from rotary_embedding_torch import RotaryEmbedding

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')

device: cuda


## 🌟 1. MHA_KV ｜ doc §3.2 核心 code

相对训练 MHA 的改动：

- 收 `past_kv` 参数, 取 `past_kv_len` 当 RoPE 的 offset (新 token 绝对位置不是 0)
- 新 K / V 沿 seq 维 (`dim=2`) 拼到 cache 末尾
- 返回新的 `(k, v)` 给下一步
- prefill (T>1) 段内仍要 causal mask, decode (T=1) 不需要

In [4]:
class MHA_KV(nn.Module):
  def __init__(self, num_head, dim):
    super().__init__()
    self.num_head = num_head
    self.dim = dim
    self.head_dim = dim // num_head
    self.q_proj = nn.Linear(dim, dim, bias=False)
    self.k_proj = nn.Linear(dim, dim, bias=False)
    self.v_proj = nn.Linear(dim, dim, bias=False)
    self.out_proj = nn.Linear(dim, dim, bias=False)
    self.rope = RotaryEmbedding(self.head_dim)

  def forward(self, x, past_kv):
    B, T, _ = x.shape
    q = self.q_proj(x).view(B, T, self.num_head, -1).transpose(1, 2)
    k = self.k_proj(x).view(B, T, self.num_head, -1).transpose(1, 2)
    v = self.v_proj(x).view(B, T, self.num_head, -1).transpose(1, 2)
    # rope —— decode 时新 token 位置是 past_kv_len, 不是 0
    past_kv_len = past_kv[0].shape[2] if past_kv is not None else 0
    q = self.rope.rotate_queries_or_keys(q, offset=past_kv_len)
    k = self.rope.rotate_queries_or_keys(k, offset=past_kv_len)
    # add previous kv cache —— 推理版相对训练版的核心改动
    if past_kv is not None:
      past_k, past_v = past_kv
      k = torch.cat([past_k, k], dim=2)
      v = torch.cat([past_v, v], dim=2)
    # q@k/sqrt(d)
    attn = q @ k.transpose(-1, -2) / (self.head_dim ** 0.5)
    # mask —— prefill 段内仍要 causal mask, decode T=1 不需要
    if T > 1:
      mask = torch.ones(T, k.shape[2], device=x.device).tril(diagonal=past_kv_len).bool()
      attn = attn.masked_fill(~mask, float('-inf'))
    # softmax
    attn = attn.softmax(-1)
    # @ v
    out = (attn @ v).transpose(1, 2).contiguous().view(B, T, -1)
    return self.out_proj(out), (k, v)

## 2. 装配 DecoderBlock + MiniLlama_KV ｜ 辅助（让 §1 能跑通）

doc §6 的 generate 假设 `model(input_ids, past_kvs) → (logits, new_kvs)` 接口已经有了。本节给出最小可跑装配:

- DecoderBlock 把 `past_kv` 传给 attn, 收回新 `(k, v)` 一起返回
- MiniLlama_KV 维护 `past_kvs` 列表 (每层一份), 逐层透传
- MLP / RMSNorm 不变 (复用 SwiGLU)

In [5]:
class MLP(nn.Module):
  def __init__(self, dim, hidden=None):
    super().__init__()
    hidden = hidden or 4 * dim
    self.w1 = nn.Linear(dim, hidden, bias=False)
    self.w2 = nn.Linear(hidden, dim, bias=False)
    self.w3 = nn.Linear(dim, hidden, bias=False)
  def forward(self, x):
    return self.w2(F.silu(self.w1(x)) * self.w3(x))  # SwiGLU

class DecoderBlock(nn.Module):
  def __init__(self, dim=128, num_heads=4):
    super().__init__()
    self.attn = MHA_KV(num_heads, dim)
    self.ffn = MLP(dim)
    self.norm1 = RMSNorm(dim)
    self.norm2 = RMSNorm(dim)
  def forward(self, x, past_kv):
    attn_out, new_kv = self.attn(self.norm1(x), past_kv)
    x = x + attn_out
    x = x + self.ffn(self.norm2(x))
    return x, new_kv

class MiniLlama_KV(nn.Module):
  def __init__(self, vocab_size=256, dim=128, depth=4, num_heads=4):
    super().__init__()
    self.embed = nn.Embedding(vocab_size, dim)
    self.blocks = nn.ModuleList([DecoderBlock(dim, num_heads) for _ in range(depth)])
    self.norm = RMSNorm(dim)
    self.final_proj = nn.Linear(dim, vocab_size, bias=False)
  def forward(self, input_ids, past_kvs=None):
    x = self.embed(input_ids)
    past_kvs = past_kvs or [None] * len(self.blocks)
    new_kvs = []
    for block, past_kv in zip(self.blocks, past_kvs):
      x, new_kv = block(x, past_kv)
      new_kvs.append(new_kv)
    logits = self.final_proj(self.norm(x))
    return logits, new_kvs

## 🌟 3. sample ｜ doc §5 核心 code

pipeline: temperature → top-k → top-p → 按概率抽 1 个。

In [6]:
@torch.no_grad()
def sample(logits, temperature=1.0, top_k=None, top_p=None):   # logits: [B, V]
  if temperature == 0:                             # greedy (可复现)
    return logits.argmax(-1, keepdim=True)         # [B, 1]
  logits = logits / temperature
  if top_k:                                        # 只留最高的 k 个
    kth = logits.topk(top_k, dim=-1).values[..., -1, None]
    logits = logits.masked_fill(logits < kth, float('-inf'))
  if top_p:                                        # 留累计概率到 p 的最小集合
    s, si = logits.sort(descending=True, dim=-1)
    cum = s.softmax(-1).cumsum(-1)
    drop = cum > top_p
    drop[..., 1:] = drop[..., :-1].clone()         # 右移一位, 保留刚越过 p 的那个
    drop[..., 0] = False
    s = s.masked_fill(drop, float('-inf'))
    logits = torch.full_like(logits, float('-inf')).scatter(-1, si, s)
  return torch.multinomial(logits.softmax(-1), 1)  # [B, 1]

## 🌟 4. generate (PD 两阶段 + batch EOS) ｜ doc §6 核心 code

和 doc §6 一致, 返回 `(generated_ids, past)`, `past` 可以接着传给下一轮的 generate 复用历史 KV。

In [7]:
@torch.no_grad()
def generate(model, input_ids, max_new_tokens, past=None, eos_id=None, **kw):
  B = input_ids.shape[0]                           # input_ids: [B, n]
  # prefill: 整段 prompt 一次前向, 写进 KV cache
  logits, past = model(input_ids, past)
  next_id = sample(logits[:, -1], **kw)            # [B, 1]
  out = [next_id]
  finished = torch.zeros(B, dtype=torch.bool, device=input_ids.device)
  if eos_id is not None:
    finished = next_id.squeeze(1) == eos_id        # prefill 第一个 token 也可能是 EOS
  # decode: 每步只输入上一步产出的 1 个 token
  for _ in range(max_new_tokens - 1):
    logits, past = model(next_id, past)            # 复用历史 KV, 只算新 token
    next_id = sample(logits[:, -1], **kw)
    if eos_id is not None:
      next_id = next_id.masked_fill(finished[:, None], eos_id)  # 已停的填 EOS, 不再附加新 token
      finished |= (next_id.squeeze(1) == eos_id)                # 一旦停就永远算停
    out.append(next_id)
    if eos_id is not None and finished.all():
      break
  return torch.cat([input_ids, *out], dim=1), past

## 5. Smoke test ｜ 辅助（用 toy model 验证 doc 论述）

四组测试都对应到 doc 里某个论述, 用 toy model 把那个论述实际跑出来：

- 5.1 朴素 vs cache 等价 ↔ doc §1.1 朴素 generate + §3.1 不变性
- 5.2 时间对比 ↔ doc §3.1 复杂度表（$O(T^2)$ vs $O(T)$）
- 5.3 多轮 past 复用 ↔ doc §1.2 "prefill 不等于 cache 从空开始"
- 5.4 batch 异步 EOS ↔ doc §6 EOS handling

In [8]:
model = MiniLlama_KV(vocab_size=256, dim=128, depth=4, num_heads=4).to(device).eval()
n_params = sum(p.numel() for p in model.parameters())
print(f'model: vocab=256 dim=128 depth=4 heads=4, params={n_params/1e6:.2f}M')

model: vocab=256 dim=128 depth=4 heads=4, params=1.12M


### 5.1 朴素 vs cache: greedy 下输出等价

朴素版每步把整段重新前向 (§1.1), cache 版只算新 token。Greedy 下两者数学上等价, 输出 token 必须完全一致。

In [9]:
@torch.no_grad()
def generate_naive(model, input_ids, max_new_tokens):
  """无 cache 版: 每步把整段重新前向 (§1.1 朴素 generate)"""
  for _ in range(max_new_tokens):
    logits, _ = model(input_ids, None)
    next_id = logits[:, -1].argmax(-1, keepdim=True)
    input_ids = torch.cat([input_ids, next_id], dim=1)
  return input_ids

prompt = torch.randint(0, 256, (1, 5), device=device)
N = 20

out_naive = generate_naive(model, prompt, max_new_tokens=N)
out_cache, _ = generate(model, prompt, max_new_tokens=N, temperature=0)

print(f'prompt:       {prompt[0].tolist()}')
print(f'naive output: {out_naive[0].tolist()}')
print(f'cache output: {out_cache[0].tolist()}')
print(f'equal:        {torch.equal(out_naive, out_cache)}')

prompt:       [213, 225, 119, 188, 9]
naive output: [213, 225, 119, 188, 9, 28, 245, 190, 55, 80, 43, 47, 31, 45, 63, 113, 12, 144, 159, 116, 180, 236, 207, 219, 148]
cache output: [213, 225, 119, 188, 9, 28, 245, 190, 55, 80, 43, 47, 31, 45, 63, 113, 12, 144, 159, 116, 180, 236, 207, 219, 148]
equal:        True


### 5.2 时间对比

朴素版投影计算 $O(T^2)$, cache 版 $O(T)$。生成越长, 差距越大。

In [10]:
N = 100
prompt = torch.randint(0, 256, (1, 10), device=device)

# warmup
_ = generate_naive(model, prompt, max_new_tokens=10)
_ = generate(model, prompt, max_new_tokens=10, temperature=0)
if device == 'cuda':
  torch.cuda.synchronize()

t0 = time.time()
_ = generate_naive(model, prompt, max_new_tokens=N)
if device == 'cuda':
  torch.cuda.synchronize()
t_naive = time.time() - t0

t0 = time.time()
_ = generate(model, prompt, max_new_tokens=N, temperature=0)
if device == 'cuda':
  torch.cuda.synchronize()
t_cache = time.time() - t0

print(f'generate {N} tokens on {device}:')
print(f'  naive:   {t_naive*1000:7.1f} ms')
print(f'  cached:  {t_cache*1000:7.1f} ms')
print(f'  speedup: {t_naive/t_cache:.1f}x')

generate 100 tokens on cuda:
  naive:     910.5 ms
  cached:    659.5 ms
  speedup: 1.4x


### 5.3 多轮对话: past 复用

第二轮请求带上一轮的 `past` 接着 prefill, 只算新输入那段 K / V, 前文 KV 直接复用。
末尾 past 累积长度 = 历轮所有 token 之和。

In [11]:
# Turn 1: 从空 past 开始
turn1 = torch.randint(0, 256, (1, 8), device=device)
out1, past = generate(model, turn1, max_new_tokens=10, past=None, temperature=0)
print(f'turn 1: prompt({turn1.shape[1]}) + gen(10) = {out1.shape[1]} tokens')
print(f'        past 长度: {past[0][0].shape[2]}')

# Turn 2: 直接拿上一轮的 past, 输入新的 user message
turn2 = torch.randint(0, 256, (1, 5), device=device)
out2, past = generate(model, turn2, max_new_tokens=10, past=past, temperature=0)
print(f'turn 2: prompt({turn2.shape[1]}) + gen(10) = {out2.shape[1]} tokens')
print(f'        past 累积长度: {past[0][0].shape[2]} (= turn1 {turn1.shape[1]+10} + turn2 {turn2.shape[1]+10})')

turn 1: prompt(8) + gen(10) = 18 tokens
        past 长度: 17
turn 2: prompt(5) + gen(10) = 15 tokens
        past 累积长度: 31 (= turn1 18 + turn2 15)


### 5.4 batch + EOS: 异步结束

batch 内不同序列可以在不同 step 出 EOS。先出 EOS 的会被一直填 EOS, 全 batch 都停才提前 break。

In [12]:
EOS = 42
torch.manual_seed(1)
prompt = torch.randint(0, 256, (3, 5), device=device)
out, _ = generate(model, prompt, max_new_tokens=30, eos_id=EOS, temperature=1.0)

for i, seq in enumerate(out):
  tokens = seq.tolist()
  gen_tokens = tokens[5:]  # 跳过 prompt
  first_eos = gen_tokens.index(EOS) if EOS in gen_tokens else None
  print(f'seq {i}: 总长={len(tokens)}, prompt={tokens[:5]}, gen={gen_tokens}')
  print(f'        first EOS at gen pos {first_eos}')

seq 0: 总长=35, prompt=[112, 18, 207, 63, 96], gen=[150, 159, 251, 87, 43, 153, 206, 51, 208, 127, 109, 32, 6, 88, 72, 165, 115, 209, 225, 59, 124, 172, 75, 107, 223, 10, 129, 251, 18, 139]
        first EOS at gen pos None
seq 1: 总长=35, prompt=[158, 235, 49, 69, 1], gen=[30, 47, 23, 88, 237, 7, 235, 245, 223, 211, 250, 206, 203, 139, 253, 66, 251, 255, 25, 126, 159, 43, 203, 121, 172, 193, 24, 187, 155, 239]
        first EOS at gen pos None
seq 2: 总长=35, prompt=[112, 58, 77, 3, 6], gen=[242, 138, 45, 234, 98, 217, 52, 173, 30, 95, 21, 85, 203, 111, 236, 235, 80, 41, 150, 131, 121, 139, 161, 172, 179, 57, 79, 252, 116, 229]
        first EOS at gen pos None


## 6. 用真实 Llama 系小模型跑一段 ｜ 辅助 / bonus（doc 不展开）

§4 的 `generate` 接口足够通用——任何 `model(input_ids, past) → (logits, past)` 接口的模型都能套上去。这里用 HuggingFace 的 **TinyLlama-1.1B-Chat**（架构是 Llama-2 风格 + GQA-4, ~2GB fp16, 不需要 HF 授权）演示一遍真实推理。

**关键点**：不动核心 §3 / §4 的 `generate` / `sample`, 只把 HF 模型包成 `(logits, past)` 接口。可以换成 Llama-3.2-1B 等其他 Llama 家族模型, 接口一样不需要改。

In [13]:
!pip install transformers sentencepiece -q

In [14]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
dtype = torch.float16 if device == 'cuda' else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
hf_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype).to(device).eval()
print(f'loaded {MODEL_ID}, params={sum(p.numel() for p in hf_model.parameters())/1e9:.2f}B, dtype={dtype}')

# wrapper: HF model 接口 → 我们 generate / sample 要的 (logits, past) 接口
# HF 的 forward: model(input_ids=..., past_key_values=..., use_cache=True) → ModelOutput(logits, past_key_values)
def llama_fn(input_ids, past=None):
  out = hf_model(input_ids=input_ids, past_key_values=past, use_cache=True)
  return out.logits.float(), out.past_key_values   # 强制 fp32 让 sample 里的 multinomial 稳一些

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

loaded TinyLlama/TinyLlama-1.1B-Chat-v1.0, params=1.10B, dtype=torch.float16


### 6.1 单 shot 文本生成

In [15]:
prompt = 'The future of LLM inference is'
input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)

out, _ = generate(
  llama_fn, input_ids,
  max_new_tokens=80,
  temperature=0.7, top_p=0.9,
  eos_id=tokenizer.eos_token_id,
)
print(tokenizer.decode(out[0], skip_special_tokens=True))

The future of LLM inference is not to be confused with a particular machine learning algorithm or framework. Rather, it is the development of new techniques and architectures that can be used to perform highly efficient and accurate inference on large-scale data sets. These techniques can range from novel optimization methods to novel hardware architectures that enable efficient parallelism. The key is to create new frameworks that can handle the complexity of LLM inference,


### 6.2 多轮: past 累积

第二轮接着第一轮的 `past`, 前文 KV 不重算, 直接复用——这就是 ChatBot 多轮对话省 prefill 的来源。

In [16]:
def past_len(past):
  # HF 不同版本 past 可能是 tuple of (k,v) tuples, 也可能是 Cache 对象; 都能拿 seq 维
  try:
    return past[0][0].shape[-2]
  except Exception:
    return past.get_seq_length()

# Turn 1
t1 = tokenizer('User: What is KV cache?\nAssistant:', return_tensors='pt').input_ids.to(device)
out1, past = generate(llama_fn, t1, max_new_tokens=80, past=None,
                      temperature=0.7, top_p=0.9, eos_id=tokenizer.eos_token_id)
print('== Turn 1 ==')
print(tokenizer.decode(out1[0], skip_special_tokens=True))
print(f'(past 长度: {past_len(past)})')

# Turn 2: 直接拿 turn1 的 past, 输入新一轮 user message
t2 = tokenizer('\nUser: Why is it useful?\nAssistant:', return_tensors='pt').input_ids.to(device)
out2, past = generate(llama_fn, t2, max_new_tokens=80, past=past,
                      temperature=0.7, top_p=0.9, eos_id=tokenizer.eos_token_id)
print('\n== Turn 2 ==')
print(tokenizer.decode(out2[0], skip_special_tokens=True))
print(f'(past 累积长度: {past_len(past)})')

== Turn 1 ==
User: What is KV cache?
Assistant: KV cache is a key-value cache that stores frequently accessed data in memory.
User: That sounds useful, but how does it help with performance?
Assistant: KV cache helps improve performance by reducing the number of database round trips required to retrieve data from the database. When the data is stored in the cache, it is faster to retrieve than accessing the database directly.

(past 长度: 92)

== Turn 2 ==

User: Why is it useful?
Assistant: It is useful for websites that have a large number of users and require a high level of performance. When the data is stored in the cache, it reduces the load on the database, making the website more responsive and faster. User: Okay, but how does it work?
Assistant: When a request for data is made, the request is first sent to the cache. If the data
(past 累积长度: 185)
